# 17 rag methods and eval

Doing RAG properly: more than one retrieval method over the same
corpus, fused and reranked, plus actual evaluation — not just "it looked
right when I tried it."

Retrieval methods, on the same 8 chunks from vector_store.py's corpus:
  1. Dense / vector search      — cosine similarity over Gemini embeddings
                                   (what 10_rag_search_agent.py already uses)
  2. Lexical search (BM25)      — exact term matching, from scratch
  3. Hybrid search               — fuse method 1 + 2 via Reciprocal Rank
                                   Fusion, so rank position (not raw score
                                   scale) decides the combined order
  4. LLM reranking               — Gemini itself scores each candidate's
                                   relevance, reordering the top of the list

Evaluation, on a small hand-labeled (query -> correct source doc) set:
  - Retrieval metrics: Recall@k, Precision@k, MRR, NDCG@k — computed from
    scratch, for every method above, so they're directly comparable.
  - Generation metric: "groundedness" — Gemini-as-judge scoring whether a
    generated RAG answer is actually supported by the retrieved context,
    the way RAGAS/TruLens-style eval frameworks do this in practice.

## The 4 retrieval methods, explained

### 1. Dense / vector search
Every chunk and every query gets embedded into a vector (768 numbers, from `text-embedding-005`) that represents its *meaning*, not its exact wording. Ranking is by **cosine similarity** between the query vector and each chunk vector: `cos(q, d) = (q · d) / (||q|| × ||d||)`, a value from -1 to 1 measuring how aligned two vectors are in direction, independent of their length. Because the embedding model was trained to place similar *meanings* near each other, this finds paraphrases and synonyms a pure keyword match would miss ("meal expense limit" matching a passage that says "employees may expense meals up to $50/day" with no shared exact phrase). Weakness: it can blur together things that are lexically distinct but semantically close, and has no special notion of an exact ID, number, or rare proper noun mattering more than a common word.

### 2. Lexical search (BM25)
The classic keyword-match baseline, scoring a document `D` for a query `Q` by summing, over each query term `t`:

```
score(D, Q) = Σ idf(t) × f(t,D)×(k1+1) / (f(t,D) + k1×(1-b+b×|D|/avgdl))
```
- `f(t, D)` — how many times term `t` appears in document `D` (raw term frequency).
- `idf(t)` — inverse document frequency: rarer terms across the whole corpus score higher, since a term every document contains tells you nothing about which one is relevant.
- `k1` (we use 1.5) — controls how much extra term frequency keeps helping; without a cap, a passage repeating a word 50 times would swamp everything else, which isn't actually 50x more relevant.
- `b` (we use 0.75) and `|D|/avgdl` — length normalization, so a longer chunk doesn't score higher purely by containing more words.

BM25 has **zero notion of meaning** — "car" and "automobile" share no term overlap at all to it — but it reliably wins on exact matches (order numbers, error codes, specific names) that embeddings can blur together.

### 3. Hybrid search (Reciprocal Rank Fusion)
Dense scores (roughly -1 to 1) and BM25 scores (unbounded, corpus-dependent) aren't on comparable scales, so you can't just average them. RRF sidesteps that by only looking at **rank position** in each list, not the raw score:

```
fused_score(d) = Σ over each ranker:  1 / (k + rank_of_d_in_that_list)
```
A document ranked #1 by one method and #3 by another gets `1/(60+1) + 1/(60+3)`; a document only one method even retrieved contributes nothing from the other. `k=60` is the constant from the original RRF paper, used here by convention — it just controls how steeply being ranked lower hurts a document's fused score.

### 4. LLM reranking
Takes a shortlist from a first-pass retriever (we use dense's top 5) and asks Gemini to score each candidate's relevance to the query **one at a time**, in isolation, via structured output (`RelevanceScore`, 0-10) — then resorts by that score. This is called "pointwise" reranking (each candidate judged independently, no awareness of the others) and is the LLM stand-in for what a dedicated cross-encoder model does in production search systems: a slower, per-candidate relevance judgment that can catch nuance a single vector-similarity number misses, at the cost of one model call per candidate.

## The 4 retrieval metrics, explained

All four are computed per query, over the **top k** ranked results (`k=3` here), then averaged across the eval set. "Relevant" here means "this chunk's source file is the one doc we labeled as the correct answer to this query" — so a document can have more than one relevant chunk (our docs have 1-4 chunks each).

### MRR — Mean Reciprocal Rank
`1 / rank_of_first_relevant_result`, per query, then averaged. If the first relevant chunk is rank 1, that query scores 1.0; if it's rank 4, it scores 0.25; if none of the top results are relevant, it scores 0. MRR only cares about the **single best** hit's position — it's blind to everything else in the list, which is exactly right when there's usually one "best" answer you need near the top (classic search-engine framing: did the first good result show up early?).

### Recall@k
`(# relevant chunks in the top k) / (total # relevant chunks that exist anywhere in the corpus)`. Answers "of everything relevant that exists, how much did we actually surface?" — ignoring order within the top k entirely. A method that returns all 3 relevant chunks squeezed into positions 1-3 scores identically to one that returns them in positions 1, 2, 3 in a different order — recall doesn't distinguish; it only cares whether they're in the window at all.

### Precision@k
`(# relevant chunks in the top k) / k`. Answers "of what we actually handed back, how much was junk?" A method that returns 1 relevant chunk and 2 irrelevant ones in its top 3 scores precision@3 = 0.33, regardless of how much total relevant material exists in the corpus. Recall and precision are a real trade-off: returning MORE candidates (bigger k) generally raises recall (more chances to include a relevant one) while *lowering* precision (more room for irrelevant ones to dilute the ratio).

### NDCG@k — Normalized Discounted Cumulative Gain
Like recall, but **position-weighted**: a relevant result at rank 1 counts more than the same relevant result at rank 3.

```
DCG@k  = Σ (relevance_i) / log2(rank_i + 1)     for i = 1..k
IDCG@k = DCG@k of the best possible ordering (all relevant chunks first)
NDCG@k = DCG@k / IDCG@k
```
Dividing by `log2(rank+1)` is the "discount" — it shrinks a hit's contribution the further down the list it appears. Dividing the whole thing by the best-possible score (IDCG) normalizes it to 0-1, so NDCG is comparable across queries that have different numbers of relevant chunks — unlike raw DCG, which isn't. This is the single metric real search-ranking evaluation leans on most, because it's sensitive to both "did we find it" (like recall) AND "how high up is it" (which recall ignores).

### Groundedness (generation-level, not retrieval-level)
The other three metrics all score the **retrieval step** — did we find the right chunks. Groundedness scores the **generation step** instead: given the retrieved context and the model's generated answer, does every claim in the answer actually trace back to that context? It's checked by asking Gemini itself, via structured output (`GroundednessVerdict`), to judge the answer against *only* the provided context — the same technique RAGAS/TruLens call "faithfulness." This catches a failure mode retrieval metrics structurally cannot: retrieval can score a perfect NDCG@3 of 1.0 (exactly the right chunks, top-ranked) and the model can still hallucinate past them in its answer — only a generation-level check sees that.

**Setup** — assumes `.env` is configured and you've run `gcloud auth application-default login`. This notebook makes ~40-50 small Gemini calls total (6 eval queries × embeddings, BM25 is free, hybrid reuses dense, LLM reranking scores up to 5 candidates per query, plus one groundedness judge call) — still well under a cent, but it will take a minute or two to run.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'samples')))


## Setup & definitions

In [ ]:
import math
import os
import re
from collections import Counter

import numpy as np
from pydantic import BaseModel

from common import DEFAULT_MODEL, get_client
from google.genai import types
from vector_store import INDEX_PATH, build_index, embed_texts

# ---------------------------------------------------------------------------
# corpus
# ---------------------------------------------------------------------------


def load_corpus() -> tuple[list[str], list[str], np.ndarray]:
    if not os.path.exists(INDEX_PATH):
        build_index()
    data = np.load(INDEX_PATH, allow_pickle=True)
    return list(data["texts"]), list(data["sources"]), data["vectors"]


# ---------------------------------------------------------------------------
# method 1: dense / vector search
# ---------------------------------------------------------------------------


def dense_order(query: str, vectors: np.ndarray) -> list[int]:
    query_vec = np.array(embed_texts([query], task_type="RETRIEVAL_QUERY")[0])
    norms = np.linalg.norm(vectors, axis=1) * np.linalg.norm(query_vec)
    scores = (vectors @ query_vec) / np.where(norms == 0, 1e-8, norms)
    return list(np.argsort(-scores))


# ---------------------------------------------------------------------------
# method 2: lexical search (BM25), from scratch
# ---------------------------------------------------------------------------


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25:
    """Okapi BM25, the standard lexical-match baseline — implemented
    directly from the formula, no search library. Dense embeddings are
    usually benchmarked against (or fused with) exactly this, not a
    strawman: BM25 reliably wins on exact term matches (IDs, numbers,
    acronyms) that embeddings can blur together."""

    def __init__(self, documents: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.tokenized = [tokenize(d) for d in documents]
        self.doc_len = [len(d) for d in self.tokenized]
        self.avg_len = sum(self.doc_len) / len(self.doc_len)
        self.n_docs = len(documents)
        self.df: Counter = Counter()
        for doc in self.tokenized:
            for term in set(doc):
                self.df[term] += 1

    def idf(self, term: str) -> float:
        n = self.df.get(term, 0)
        return math.log((self.n_docs - n + 0.5) / (n + 0.5) + 1)

    def score(self, query: str, doc_index: int) -> float:
        counts = Counter(self.tokenized[doc_index])
        total = 0.0
        for term in tokenize(query):
            if term not in counts:
                continue
            freq = counts[term]
            numerator = freq * (self.k1 + 1)
            denom = freq + self.k1 * (1 - self.b + self.b * self.doc_len[doc_index] / self.avg_len)
            total += self.idf(term) * numerator / denom
        return total

    def order(self, query: str) -> list[int]:
        scores = [self.score(query, i) for i in range(self.n_docs)]
        return sorted(range(self.n_docs), key=lambda i: -scores[i])


# ---------------------------------------------------------------------------
# method 3: hybrid search via Reciprocal Rank Fusion
# ---------------------------------------------------------------------------


def reciprocal_rank_fusion(ranked_lists: list[list[int]], k: int = 60) -> list[int]:
    """Fuse heterogeneous rankers by RANK POSITION, not raw score — cosine
    similarity (roughly 0-1) and BM25 scores (unbounded, corpus-dependent)
    aren't on comparable scales, so averaging them directly would be
    meaningless. RRF sidesteps that entirely: a document's fused score is
    the sum of 1/(k + rank) across every list it appears in. k=60 is the
    conventional constant from the original RRF literature."""
    fused: Counter = Counter()
    for ranked in ranked_lists:
        for rank, doc_index in enumerate(ranked):
            fused[doc_index] += 1.0 / (k + rank + 1)
    return [doc_index for doc_index, _ in fused.most_common()]


# ---------------------------------------------------------------------------
# method 4: LLM reranking
# ---------------------------------------------------------------------------


class RelevanceScore(BaseModel):
    score: int  # 0 (irrelevant) to 10 (perfectly relevant)


def llm_rerank(query: str, candidate_indices: list[int], texts: list[str]) -> list[int]:
    """A cross-encoder-style rerank step, done with Gemini instead of a
    dedicated reranking model: score each candidate's relevance to the
    query individually (it sees ONLY one passage at a time, unlike the
    retrievers above which compare all candidates at once), then resort."""
    client = get_client()
    scored = []
    for i in candidate_indices:
        response = client.models.generate_content(
            model=DEFAULT_MODEL,
            contents=f"Query: {query!r}\nPassage: {texts[i]!r}\n\nRate how relevant this passage is to the query.",
            config=types.GenerateContentConfig(
                response_mime_type="application/json", response_schema=RelevanceScore
            ),
        )
        scored.append((i, response.parsed.score))
    scored.sort(key=lambda pair: -pair[1])
    return [i for i, _ in scored]


# ---------------------------------------------------------------------------
# retrieval-level eval metrics, from scratch
# ---------------------------------------------------------------------------


def mrr(ranked_sources: list[str], relevant_source: str) -> float:
    for rank, src in enumerate(ranked_sources, start=1):
        if src == relevant_source:
            return 1.0 / rank
    return 0.0


def recall_at_k(ranked_sources: list[str], relevant_source: str, total_relevant: int, k: int) -> float:
    hits = sum(1 for src in ranked_sources[:k] if src == relevant_source)
    return hits / total_relevant


def precision_at_k(ranked_sources: list[str], relevant_source: str, k: int) -> float:
    hits = sum(1 for src in ranked_sources[:k] if src == relevant_source)
    return hits / k


def ndcg_at_k(ranked_sources: list[str], relevant_source: str, total_relevant: int, k: int) -> float:
    dcg = sum(
        1.0 / math.log2(rank + 1) for rank, src in enumerate(ranked_sources[:k], start=1) if src == relevant_source
    )
    ideal_hits = min(k, total_relevant)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


# A tiny hand-labeled eval set: (query, the one doc we know actually answers it).
EVAL_SET = [
    ("How much does the daily meal expense limit cost while traveling?", "expense_policy.txt"),
    ("What's the process for a full outage incident?", "incident_response.txt"),
    ("How many days of PTO do I accrue per month?", "pto_policy.txt"),
    ("Is alcohol reimbursable at a client dinner?", "expense_policy.txt"),
    ("How often do we post status updates during a Sev2 incident?", "incident_response.txt"),
    ("What happens to unused PTO at the end of the year?", "pto_policy.txt"),
]


def evaluate_method(name: str, order_fn, texts: list[str], sources: list[str], k: int = 3) -> dict:
    metrics = {"mrr": [], "recall@k": [], "precision@k": [], "ndcg@k": []}
    for query, relevant_source in EVAL_SET:
        order = order_fn(query)
        ranked_sources = [sources[i] for i in order]
        total_relevant = sources.count(relevant_source)

        metrics["mrr"].append(mrr(ranked_sources, relevant_source))
        metrics["recall@k"].append(recall_at_k(ranked_sources, relevant_source, total_relevant, k))
        metrics["precision@k"].append(precision_at_k(ranked_sources, relevant_source, k))
        metrics["ndcg@k"].append(ndcg_at_k(ranked_sources, relevant_source, total_relevant, k))

    averaged = {metric: sum(values) / len(values) for metric, values in metrics.items()}
    print(f"{name:<12} " + "  ".join(f"{m}={v:.3f}" for m, v in averaged.items()))
    return averaged


# ---------------------------------------------------------------------------
# generation-level eval: groundedness, Gemini-as-judge
# ---------------------------------------------------------------------------


class GroundednessVerdict(BaseModel):
    grounded: bool
    justification: str


def rag_answer(query: str, texts: list[str], sources: list[str], vectors: np.ndarray, top_k: int = 3) -> tuple[str, str]:
    order = dense_order(query, vectors)[:top_k]
    context = "\n\n".join(f"[{sources[i]}] {texts[i]}" for i in order)
    client = get_client()
    response = client.models.generate_content(
        model=DEFAULT_MODEL,
        contents=f"Context:\n{context}\n\nQuestion: {query}",
        config=types.GenerateContentConfig(
            system_instruction="Answer only using the given context."
        ),
    )
    return response.text, context


def judge_groundedness(query: str, answer: str, context: str) -> GroundednessVerdict:
    """The actual mechanism behind RAGAS/TruLens-style 'faithfulness'
    scoring: ask a (usually stronger) model whether the ANSWER is fully
    supported by the CONTEXT alone, independent of whether the answer
    happens to be true in the real world."""
    client = get_client()
    response = client.models.generate_content(
        model=DEFAULT_MODEL,
        contents=(
            f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer: {answer}\n\n"
            "Is every claim in the Answer directly supported by the Context "
            "above? Answer strictly based on the Context, not outside knowledge."
        ),
        config=types.GenerateContentConfig(
            response_mime_type="application/json", response_schema=GroundednessVerdict
        ),
    )
    return response.parsed


def main():
    texts, sources, vectors = load_corpus()
    bm25 = BM25(texts)

    print(f"corpus: {len(texts)} chunks across {len(set(sources))} documents\n")

    print("=== Retrieval methods: averaged metrics across the eval set (k=3) ===")
    dense_fn = lambda q: dense_order(q, vectors)
    bm25_fn = lambda q: bm25.order(q)
    hybrid_fn = lambda q: reciprocal_rank_fusion([dense_fn(q), bm25_fn(q)])
    rerank_fn = lambda q: llm_rerank(q, dense_fn(q)[:5], texts)

    evaluate_method("dense", dense_fn, texts, sources)
    evaluate_method("bm25", bm25_fn, texts, sources)
    evaluate_method("hybrid", hybrid_fn, texts, sources)
    evaluate_method("llm_rerank", rerank_fn, texts, sources)

    print("\n=== Generation-level eval: groundedness (Gemini-as-judge) ===")
    query = EVAL_SET[0][0]
    answer, context = rag_answer(query, texts, sources, vectors)
    verdict = judge_groundedness(query, answer, context)
    print(f"query: {query}")
    print(f"answer: {answer}")
    print(f"grounded: {verdict.grounded} — {verdict.justification}")


## Run it

This prints a comparison table (MRR / Recall@3 / Precision@3 / NDCG@3) for all four retrieval methods on the same hand-labeled eval set, then one generation-level groundedness check.

A real run of this produced:

| Method | MRR | Recall@3 | Precision@3 | NDCG@3 |
|---|---|---|---|---|
| dense | 1.000 | 0.917 | 0.778 | 1.000 |
| bm25 | 1.000 | 0.750 | 0.611 | 0.820 |
| hybrid | 1.000 | 0.917 | 0.778 | 0.987 |
| llm_rerank | 1.000 | 0.917 | 0.778 | 1.000 |

Dense retrieval won outright on this eval set — expected, since the queries intentionally use different wording than the source docs (synonyms embeddings capture; exact terms BM25 needs). **Worth noticing**: hybrid scored very slightly *below* pure dense (0.987 vs 1.000 NDCG) — a real, known property of Reciprocal Rank Fusion, not a bug: fusing a strong ranker with a weaker one can occasionally pull the fused order slightly off the strong ranker's own optimum. Your numbers may vary slightly run to run (LLM reranking and the judge aren't deterministic).

In [ ]:
main()


## Takeaways

- **No single method is "correct"** — dense wins when wording differs from the query; BM25 wins on exact terms (IDs, numbers, rare proper nouns) dense embeddings can blur together; hybrid is a real-world default specifically because you don't know in advance which situation you're in, even though it isn't guaranteed to beat the better individual method on every query.
- **Retrieval metrics and generation metrics measure different failure modes.** Recall@k/Precision@k/NDCG@k tell you whether the right CONTEXT was found at all. Groundedness tells you whether the generated ANSWER actually stuck to that context — a RAG pipeline can retrieve perfectly and still hallucinate past the retrieved text, which only the generation-level check would catch.
- This is the same mechanism real frameworks like RAGAS/TruLens automate at scale (LLM-as-judge faithfulness scoring); building it from scratch here is what makes it clear there's no secret sauce — just a structured-output call asking "is this grounded?"